In [ ]:
# importing the libraries
import torch, torchvision, os, PIL, pdb
from torch import nn
import torch.distributed as dist
import torch.multiprocessing as mp
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.utils.data import Dataset
from torch.utils.data import DataLoader
from torchvision import transforms
from torchvision.utils import make_grid
from  tqdm.auto import tqdm
import numpy as np
import PIL
import matplotlib.pyplot as plt
import os

def show(tensor, num=25, wandbactive=0, name=''):
    data = tensor.detach().cpu()
    grid = make_grid(data[:num], nrow=5).permute(1,2,0)

    if (wandbactive==1):
        wandb.log({name:wandb.Image(grid.numpy().clip(0,1))})

    plt.imshow(grid.clip(0,1))
    plt.show()

# hyperparameters and general parameters
n_epochs=10000
batch_size=128 # 128
lr=1e-4
z_dim=200
device='cuda' #GPU

cur_step=0
critic_cycles=5
gen_losses=[]
crit_losses=[]
show_step=35
save_step=35

wandbact=1 # we want to track state through weights and biasses, optional

# WANDB (weghts and biases)
## Create an account on wandb.ai. This is a usuful tool to keep track of the training process remotely and compare results and loss through training across different runs. Remember to replace the placeholder in the cellbelow with your WANDB API KEY.

In [ ]:
# optional
!pip install wandb -qqq
import wandb
wandb.login(key='ADD YOUR WANDB API KEY HERE')

In [ ]:
%%capture
experiment_name = wandb.util.generate_id()

myrun= wandb.init(
    project='wgan',
    group=experiment_name,
    config={
        "optimizer":"sgd",
        "model":"wgan gp",
        "epoch":"10000",
        "batch_size":batch_size
    }
)

config=wandb.config

print(experiment_name)

In [ ]:
# generator model

class Generator(nn.Module):
    def __init__(self, z_dim=64, d_dim=16):
        super(Generator, self).__init__()
        self.z_dim=z_dim

        self.gen = nn.Sequential(
            # ConvTranspose2d : in_channels, out_channels, kernel_size, stride=1, padding=0
            # new width and height: (n-1)*stride - 2*padding + ks
            # n: width or height, ks: kernel size
            nn.ConvTranspose2d(z_dim, d_dim*32, 4, 1, 0), # 1x1x200 -> 4x4x512
            nn.BatchNorm2d(d_dim*32),
            nn.ReLU(True),

            nn.ConvTranspose2d(d_dim*32, d_dim*16, 4, 2, 1), # 4x4x512 -> 8x8x256
            nn.BatchNorm2d(d_dim*16),
            nn.ReLU(True),

            nn.ConvTranspose2d(d_dim*16, d_dim*8, 4, 2, 1), # 8x8x256 -> 16x16x128
            nn.BatchNorm2d(d_dim*8),
            nn.ReLU(True),

            nn.ConvTranspose2d(d_dim*8, d_dim*4, 4, 2, 1), # 16x16x128 -> 32x32x64
            nn.BatchNorm2d(d_dim*4),
            nn.ReLU(True),

            nn.ConvTranspose2d(d_dim*4, d_dim*2, 4, 2, 1), # 32x32x64 -> 64x64x32
            nn.BatchNorm2d(d_dim*2),
            nn.ReLU(True),

            nn.ConvTranspose2d(d_dim*2, 3, 4, 2, 1), # 64x64x32 -> 128x128x3
            nn.Tanh() #produce a result in the range [-1,1]
        )

    def forward(self, noise):
        x = noise.view(len(noise), self.z_dim ,1, 1) # size 128x200x1x1
        return self.gen(x)

def gen_noise(num, z_dim, device='cuda'):
    return torch.randn(num, z_dim, device=device)


In [ ]:
# Critic model

class Critic(nn.Module):
    def __init__(self, d_dim=16):
        super(Critic, self).__init__()

        self.crit = nn.Sequential(
            # Conv2d params: in_chanenels, out_channels, kernel_size, stride, padding
            # New width or height: (n+2*pad-ks)//stride +1
            nn.Conv2d(3, d_dim, 4, 2, 1), # 128x128x3 -> 64x64x16
            nn.InstanceNorm2d(d_dim),
            nn.LeakyReLU(0.2),

            nn.Conv2d(d_dim, d_dim*2, 4, 2, 1), # 64x64x16 -> 32x32x32
            nn.InstanceNorm2d(d_dim*2),
            nn.LeakyReLU(0.2),

            nn.Conv2d(d_dim*2, d_dim*4, 4, 2, 1), # 32x32x32 -> 16x16x64
            nn.InstanceNorm2d(d_dim*4),
            nn.LeakyReLU(0.2),

            nn.Conv2d(d_dim*4, d_dim*8, 4, 2, 1), # 16x16x64 -> 8x8x126
            nn.InstanceNorm2d(d_dim*8),
            nn.LeakyReLU(0.2),

            nn.Conv2d(d_dim*8, d_dim*16, 4, 2, 1), # 8x8x126 -> 4x4x256
            nn.InstanceNorm2d(d_dim*16),
            nn.LeakyReLU(0.2),

            nn.Conv2d(d_dim*16, 1, 4, 1, 0), # 4x4x256 -> 1x1x1
        )

    def forward(self, image):
        # image : 128x1x128x128
        crit_pred = self.crit(image) # 128x1x1x1
        return crit_pred.view(len(crit_pred),-1) # 128x1

# Note
## This document was edited using Kaggle Notebook with access to the CelebA dataset on Kaggle, which can be added using the 'Input' option on the right side of the screen. If you download and unzip the dataset, remember to assign the path to the unzipped dataset folder the variable `data_path`.

In [ ]:
# Dataset, DataLoader, declare gen, crit, test dataset

class Dataset(Dataset):
    def __init__(self, path, size=128, lim=10000):
        self.sizes=[size,size]
        items, labels = [],[]

        for data in os.listdir(path)[:lim]:
            # Eg, path is 'data/celeba/img_align_celeba', data is '112354.jpg'
            item = os.path.join(path, data)
            items.append(item)
            labels.append(data)
        self.items = items
        self.labels = labels

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        data = PIL.Image.open(self.items[idx]).convert('RGB') # (178x218)
        data = np.asarray(torchvision.transforms.Resize(self.sizes)(data)) # 128x128x3
        data = np.transpose(data, (2,0,1)).astype(np.float32, copy=False) # 3x128x128, 0->255
        data = torch.from_numpy(data).div(255) # 0->1
        return data, self.labels[idx]

## Dataset
data_path = '/kaggle/input/datasets/jessicali9530/celeba-dataset/img_align_celeba/img_align_celeba'
ds = Dataset(data_path, size=128, lim=20000) # Change the number of images you can to train on here using 'lim'

dataloader = DataLoader(ds, batch_size=batch_size, shuffle=True)

x,y=next(iter(dataloader))
# x,y=x.to(rank), y.to(rank)
show(x)

In [ ]:
# Gradient penalty calculation

def get_gp(real, fake, crit, alpha, gamma=10):
    mix_images = real*alpha + fake*(1-alpha) # 128x3x128x128
    mix_scores = crit(mix_images) # 128x1

    gradient = torch.autograd.grad(
        inputs= mix_images,
        outputs = mix_scores,
        grad_outputs = torch.ones_like(mix_scores),
        retain_graph = True,
        create_graph = True,
    )[0] # -> 128x3x128x128

    gradient = gradient.view(len(gradient), -1) # 128x49152
    gradient_norm = gradient.norm(2, dim=1)
    gp = gamma * ((gradient_norm-1)**2).mean()

    return gp

In [ ]:
# Save models and checkpoints

root_path = './data/'

if not os.path.isdir(root_path):
    os.mkdir(root_path)

def save_checkpoint(name):
    torch.save({
        'epoch': epoch,
        'model_state_dict': gen.state_dict(),
        'optimizer_state_dict': gen_opt.state_dict()
    }, f"{root_path}G-{name}.pkl")

    torch.save({
        'epoch': epoch,
        'model_state_dict': crit.state_dict(),
        'optimizer_state_dict': crit_opt.state_dict()
    }, f"{root_path}C-{name}.pkl")

    print("Saved checkpoint")

def load_checkpoint(name):
    checkpoint = torch.load(f"{root_path}G-{name}.pkl")
    gen.load_state_dict(checkpoint['model_state_dict'])
    gen_opt.load_state_dict(checkpoint['optimizer_state_dict'])

    checkpoint = torch.load(f"{root_path}C-{name}.pkl")
    crit.load_state_dict(checkpoint['model_state_dict'])
    crit_opt.load_state_dict(checkpoint['optimizer_state_dict'])

    print("Loaded checkpoint")

# Training loop on multiple GPU with DP

Training loop on multiple GPUs with DataPrarallel (DP) implementation from PyTorch. This helps make use of the two GPUs available on Kaggle Notebook to spped up training.

In [ ]:
# Training Loop

device = torch.device("cuda:0")

gen = Generator(z_dim).to(device)
crit = Critic().to(device)

if torch.cuda.device_count() > 1:
    print("Using", torch.cuda.device_count(), "GPUs")
    gen = torch.nn.DataParallel(gen)
    crit = torch.nn.DataParallel(crit)

gen_opt = torch.optim.Adam(gen.parameters(), lr=lr, betas=(0.5, 0.9))
crit_opt = torch.optim.Adam(crit.parameters(), lr=lr, betas=(0.5, 0.9))

# wandb optional
if (wandbact==1):
    wandb.watch(gen, log_freq=100)
    wandb.watch(crit, log_freq=100)

for epoch in range(n_epochs):
    for real, _ in tqdm(dataloader):
        real = real.to(device) #.cuda()
        cur_bs = len(real)

        # Critic
        mean_critic_loss = 0
        for _ in range(critic_cycles):
            crit_opt.zero_grad()

            noise = gen_noise(cur_bs, z_dim, device=device)
            fake = gen(noise)
            crit_fake_pred = crit(fake.detach())
            crit_real_pred = crit(real)

            alpha = torch.rand(len(real),1,1,1, device=device, requires_grad=True) # 128x1x1x1
            gp = get_gp(real,fake.detach(), crit, alpha)

            crit_loss = crit_fake_pred.mean() - crit_real_pred.mean() + gp

            mean_critic_loss+=crit_loss.item() / critic_cycles

            crit_loss.backward(retain_graph=True)
            crit_opt.step()

        crit_losses+=[mean_critic_loss]

        # Generator
        gen_opt.zero_grad()
        noise = gen_noise(cur_bs, z_dim, device=device)
        fake = gen(noise)
        crit_fake_pred = crit(fake)
        gen_loss = -crit_fake_pred.mean()
        gen_loss.backward()
        gen_opt.step()

        gen_losses+=[gen_loss.item()]

        # Stats
        if(wandbact==1):
            wandb.log({'Epoch': epoch, 'Step': cur_step, 'Critic loss': mean_critic_loss, 'Gen loss': gen_loss})

        if(cur_step%save_step==0 and cur_step>0):
            print('Saving checkpoint: ', cur_step, save_step)
            save_checkpoint("latest_2GPUS") # change name to include step number

        if(cur_step%show_step==0 and cur_step>0):
            show(fake, wandbactive=1, name='fake')
            show(real, wandbactive=1, name='real')

            gen_mean = sum(gen_losses[-show_step:])/show_step
            crit_mean = sum(crit_losses[-show_step:])/show_step

            print(f"Epoch {epoch}: Step {cur_step}: Generator loss: {gen_mean}, critic loss: {crit_mean}")

            plt.plot(
                range(len(gen_losses)),
                torch.Tensor(gen_losses),
                label="Generator Loss"
            )

            plt.plot(
                range(len(gen_losses)),
                torch.Tensor(crit_losses),
                label="Critic Loss"
            )

            plt.ylim(-1000, 1000)
            plt.legend()
            plt.show()

        cur_step+=1



# Training loop for multi-GPU DDP

Training loop for multi-GPU DistributedDataParallel (DDP) implementation from PyTorch in the cell below. This has not been tested yet. It has been commented out to avoid accidentally running this cell.

Use the command `torchrun --nproc-per-node=<number_of_gpus> your_training_script.py` in a terminal.

You can convert this .ipynb file to a .py and use it inplace of `your_training_script.py`.

In [ ]:
# # Training loop for multi-GPU DDP

# from torch.utils.data.distributed import DistributedSampler

# WORLD_SIZE = torch.cuda.device_count() # 2

# def setup(rank, world_size):
#     os.environ["MASTER_ADDR"] = "localhost"
#     os.environ["MASTER_PORT"] = "12355"
#     dist.init_process_group("nccl", rank=rank, world_size=world_size)

# def cleanup():
#     dist.destroy_process_group()

# def GAN_ddp(rank, world_size):
#     try:
#         print(f"Running on GPU {rank}")
#         setup(rank, world_size)

#         torch.cuda.set_device(rank)

#         # Models
#         # gen, crit = get_models_opts()
#         gen = Generator(z_dim).to(rank)
#         crit = Critic().to(rank)

#         gen = DDP(gen, device_ids=[rank])
#         crit = DDP(crit, device_ids=[rank])

#         # Optimizers
#         gen_opt = torch.optim.Adam(gen.parameters(), lr=lr, betas=(0.5, 0.9))
#         crit_opt = torch.optim.Adam(crit.parameters(), lr=lr, betas=(0.5, 0.9))

#         # Distributed Sampler for Dataloader
#         sampler = DistributedSampler(ds, num_replicas=world_size, rank=rank)

#         dataloader = DataLoader(
#             ds,
#             batch_size=batch_size,
#             sampler=sampler,
#             shuffle=True
#         )

#         for epoch in range(n_epochs):
#             sampler.set_epoch(epoch)
#             for real, _ in tqdm(dataloader):
#                 real = real.to(rank)
#                 cur_bs = len(real)

#                 # Critic
#                 mean_critic_loss = 0
#                 for _ in range(critic_cycles):
#                     crit_opt.zero_grad()

#                     noise = gen_noise(cur_bs, z_dim, device=rank)
#                     fake = gen(noise)
#                     crit_fake_pred = crit(fake.detach())
#                     crit_real_pred = crit(real)

#                     alpha = torch.randn(len(real),1,1,1, device=rank, requires_grad=True) # 128x1x1x1
#                     gp = get_gp(real,fake.detach(), crit, alpha)

#                     crit_loss = crit_fake_pred.mean() - crit_real_pred.mean() + gp

#                     mean_critic_loss+=crit_loss.item() / critic_cycles

#                     crit_loss.backward()
#                     crit_opt.step()

#                 # Generator
#                 gen_opt.zero_grad()
#                 noise = gen_noise(cur_bs, z_dim, device=rank)
#                 fake = gen(noise)
#                 crit_fake_pred = crit(fake)
#                 gen_loss = -crit_fake_pred.mean()
#                 gen_loss.backward()
#                 gen_opt.step()

#                 # Stats
#                 if rank==0:
#                     crit_losses+=[mean_critic_loss]
#                     gen_losses+=[gen_loss.item()]
#                     if(wandbact==1):
#                         wandb.log({'Epoch': epoch, 'Step': cur_step, 'Critic loss': mean_critic_loss, 'Gen loss': gen_loss})

#                     if(cur_step%save_step==0 and cur_step>0):
#                         print('Saving checkpoint: ', cur_step, save_step)
#                         save_checkpoint("latest") # change name to include step number

#                     if(cur_step%show_step==0 and cur_step>0):
#                         show(fake, wandbactive=1, name='fake')
#                         show(real, wandbactive=1, name='real')

#                     gen_mean = sum(gen_losses[-show_step:])/show_step
#                     crit_mean = sum(crit_losses[-show_step:])/show_step

#                     print(f"Epoch {epoch}: Step {cur_step}: Generator loss: {gen_mean}, critic loss: {crit_mean}")

#                     plt.plot(
#                         range(len(gen_losses)),
#                         torch.Tensor(gen_losses),
#                         label="Generator Loss"
#                     )

#                     plt.plot(
#                         range(len(gen_losses)),
#                         torch.Tensor(crit_losses),
#                         label="Critic Loss"
#                     )

#                     plt.ylim(-1000, 1000)
#                     plt.legend()
#                     plt.show()

#             cur_step+=1
#         cleanup()

#     except Exception as e:
#         print(f"Error in rank {rank}: {e}")
#         raise e

# if __name__ == "__main__":
#     mp.spawn(GAN_ddp,
#             args=(WORLD_SIZE,),
#             nprocs=WORLD_SIZE,
#             join=True)